# Brain Tumor Detection and Segmentation
**Student:** Prithwiraj Prakash (Roll No: AA.SC.P2MCA25010008)

This notebook contains the complete training pipeline for:
1. Multi-class tumor classification (Glioma, Meningioma, Pituitary, Normal) using EfficientNetB3
2. Tumor lesion segmentation using U-Net


## 1. Environment and Google Drive Setup


In [ ]:
import tensorflow as tf
import os

print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    print('No GPU detected. Switch runtime type to GPU for faster training.')
else:
    print('GPU active:', gpus[0].name)
    !nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

# Mount Google Drive to store trained weights and plots
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/brain_tumor_project'
MODELS_DIR = os.path.join(DRIVE_DIR, 'models')
PLOTS_DIR  = os.path.join(DRIVE_DIR, 'plots')
PREDS_DIR  = os.path.join(DRIVE_DIR, 'predictions')
for d in [MODELS_DIR, PLOTS_DIR, PREDS_DIR]:
    os.makedirs(d, exist_ok=True)
print(f'Drive directory: {DRIVE_DIR}')


## 2. Dataset Download


In [ ]:
!pip install kagglehub opencv-python scikit-learn -q
import kagglehub, shutil

DATA_ROOT = '/content/data/raw'
os.makedirs(DATA_ROOT, exist_ok=True)

print('Downloading brain tumor classification dataset...')
path1 = kagglehub.dataset_download('masoudnickparvar/brain-tumor-mri-dataset')
print(f'   Downloaded to: {path1}')
for item in os.listdir(path1):
    src = os.path.join(path1, item)
    dst = os.path.join(DATA_ROOT, item)
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)

print('Downloading LGG segmentation dataset...')
path2 = kagglehub.dataset_download('mateuszbuda/lgg-mri-segmentation')
print(f'   Downloaded to: {path2}')
for item in os.listdir(path2):
    src = os.path.join(path2, item)
    dst = os.path.join(DATA_ROOT, item)
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)

print('Datasets ready.')


## 3. Configuration and Parameters


In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt
from glob import glob
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
from tensorflow import keras
from tensorflow.keras import layers, Model

CLASS_NAMES = ['glioma', 'meningioma', 'notumor', 'pituitary']
NUM_CLASSES = 4
IMG_SIZE    = (224, 224)
SEG_SIZE    = (256, 256)
BATCH_SIZE  = 32
SEG_BATCH   = 16
CLF_EPOCHS  = 25
SEG_EPOCHS  = 25
LR          = 1e-4
SEED        = 42

CLF_MODEL_PATH = os.path.join(MODELS_DIR, 'classifier_efficientnet.keras')
UNET_MODEL_PATH = os.path.join(MODELS_DIR, 'unet_segmentation.keras')
print('Hyperparameters configured.')

## 4. Preprocessing and Data Pipeline


In [ ]:
def read_img(path, size):
    img = cv2.imread(path)
    if img is None: return np.zeros((*size, 3), dtype=np.float32)
    img = cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), size)
    return img.astype(np.float32) / 255.0

# Collect all image file paths
paths, labels = [], []
for split in ['Training', 'Testing']:
    for idx, cls in enumerate(CLASS_NAMES):
        c_dir = os.path.join(DATA_ROOT, split, cls)
        for ext in ['*.jpg', '*.png', '*.jpeg']:
            for p in glob(os.path.join(c_dir, ext)):
                paths.append(p)
                labels.append(idx)

paths = np.array(paths)
labels = np.array(labels, dtype=np.int32)
print(f'Total classification images found: {len(paths)}')

# Split
p_tv, p_test, l_tv, l_test = train_test_split(paths, labels, test_size=0.15, stratify=labels, random_state=SEED)
p_train, p_val, l_train, l_val = train_test_split(p_tv, l_tv, test_size=0.15/0.85, stratify=l_tv, random_state=SEED)
print(f'Train: {len(p_train)} | Val: {len(p_val)} | Test: {len(p_test)}')

# Streaming tf.data pipeline
AUTOTUNE = tf.data.AUTOTUNE

def load_tf(p_tensor, l_tensor):
    def _load(pb, lb):
        im = read_img(pb.numpy().decode('utf-8'), IMG_SIZE)
        lbl = tf.one_hot(lb, NUM_CLASSES).numpy()
        return im, lbl
    im, lbl = tf.py_function(_load, [p_tensor, l_tensor], [tf.float32, tf.float32])
    im.set_shape((*IMG_SIZE, 3))
    lbl.set_shape((NUM_CLASSES,))
    return im, lbl

def augment(im, lbl):
    im = tf.image.random_flip_left_right(im)
    im = tf.image.random_flip_up_down(im)
    im = tf.image.random_brightness(im, 0.1)
    im = tf.image.random_contrast(im, 0.85, 1.15)
    return tf.clip_by_value(im, 0.0, 1.0), lbl

train_ds = (tf.data.Dataset.from_tensor_slices((p_train, l_train))
            .shuffle(len(p_train), seed=SEED)
            .map(load_tf, num_parallel_calls=AUTOTUNE)
            .map(augment, num_parallel_calls=AUTOTUNE)
            .batch(BATCH_SIZE).prefetch(AUTOTUNE))

val_ds = (tf.data.Dataset.from_tensor_slices((p_val, l_val))
          .map(load_tf, num_parallel_calls=AUTOTUNE)
          .batch(BATCH_SIZE).prefetch(AUTOTUNE))

print('Data pipeline ready.')


## 5. Classifier Training (EfficientNetB3)


In [ ]:
def build_classifier():
    inp = keras.Input(shape=(*IMG_SIZE, 3))
    x = layers.Rescaling(255.0)(inp)
    base = keras.applications.EfficientNetB3(include_top=False, weights='imagenet', input_tensor=x)
    base.trainable = False  # Freeze backbone for Phase 1
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    return Model(inputs=inp, outputs=out)

clf_model = build_classifier()
clf_model.compile(
    optimizer=keras.optimizers.Adam(LR),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy', keras.metrics.AUC(name='auc'), keras.metrics.Precision(name='precision'), keras.metrics.Recall(name='recall')]
)

cbs_p1 = [
    keras.callbacks.ModelCheckpoint(CLF_MODEL_PATH, monitor='val_accuracy', save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]

print('Phase 1: Training classifier top layers...')
h1 = clf_model.fit(train_ds, validation_data=val_ds, epochs=12, callbacks=cbs_p1)

print('Phase 2: Fine-tuning last 30 layers...')
try:
    base_m = clf_model.get_layer('efficientnetb3')
    for layer in base_m.layers[-30:]:
        if not isinstance(layer, layers.BatchNormalization):
            layer.trainable = True
except ValueError:
    for layer in clf_model.layers[-35:]:
        if not isinstance(layer, layers.BatchNormalization):
            layer.trainable = True

clf_model.compile(
    optimizer=keras.optimizers.Adam(LR / 10),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)
h2 = clf_model.fit(train_ds, validation_data=val_ds, epochs=13, callbacks=cbs_p1)
print('Classifier training complete. Model saved to Drive.')


## 6. Classifier Evaluation


In [ ]:
best_clf = tf.keras.models.load_model(CLF_MODEL_PATH)

print(f'Loading {len(p_test)} test images for evaluation...')
X_test = np.array([read_img(p, IMG_SIZE) for p in p_test], dtype=np.float32)
y_pred_prob = best_clf.predict(X_test, batch_size=BATCH_SIZE)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = l_test

print('\n' + '='*60)
print('CLASSIFICATION REPORT')
print('='*60)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# Confusion Matrix Plot
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 6))
plt.imshow(cm, cmap='Blues')
plt.title('Confusion Matrix – Brain Tumor Classifier', fontweight='bold')
plt.colorbar()
plt.xticks(range(NUM_CLASSES), CLASS_NAMES, rotation=30)
plt.yticks(range(NUM_CLASSES), CLASS_NAMES)
plt.xlabel('Predicted')
plt.ylabel('True')
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        plt.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

# ROC Curves
y_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
plt.figure(figsize=(8, 6))
colors = ['#e74c3c', '#e67e22', '#2ecc71', '#3498db']
for i, cls in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_pred_prob[:, i])
    score = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, color=colors[i], label=f'{cls} (AUC = {score:.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.title('ROC Curves', fontweight='bold')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'roc_curves.png'), dpi=150)
plt.show()

## 7. Tumor Segmentation Model (U-Net)


In [ ]:
# Custom Dice Loss & Combined Loss
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_t = tf.reshape(tf.cast(y_true, tf.float32), [-1])
    y_p = tf.reshape(tf.cast(y_pred, tf.float32), [-1])
    inter = tf.reduce_sum(y_t * y_p)
    return (2.0 * inter + smooth) / (tf.reduce_sum(y_t) + tf.reduce_sum(y_p) + smooth)

def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)

def combined_loss(y_true, y_pred):
    return 0.5 * keras.losses.binary_crossentropy(y_true, y_pred) + 0.5 * dice_loss(y_true, y_pred)

def iou_metric(y_true, y_pred, smooth=1e-6):
    y_p = tf.cast(y_pred > 0.5, tf.float32)
    y_t = tf.reshape(tf.cast(y_true, tf.float32), [-1])
    y_p = tf.reshape(y_p, [-1])
    inter = tf.reduce_sum(y_t * y_p)
    union = tf.reduce_sum(y_t) + tf.reduce_sum(y_p) - inter
    return (inter + smooth) / (union + smooth)

# U-Net Architecture
def conv_block(x, f):
    x = layers.Conv2D(f, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(f, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    return x

def build_unet(img_size=SEG_SIZE):
    inp = keras.Input(shape=(*img_size, 3))
    # Encoder
    c1 = conv_block(inp, 64); p1 = layers.MaxPooling2D(2)(c1)
    c2 = conv_block(p1, 128); p2 = layers.MaxPooling2D(2)(c2)
    c3 = conv_block(p2, 256); p3 = layers.MaxPooling2D(2)(c3)
    c4 = conv_block(p3, 512); p4 = layers.MaxPooling2D(2)(c4)
    # Bridge
    b = conv_block(p4, 1024)
    # Decoder
    u4 = layers.Conv2DTranspose(512, 2, strides=2, padding='same')(b)
    d4 = conv_block(layers.Concatenate()([u4, c4]), 512)
    u3 = layers.Conv2DTranspose(256, 2, strides=2, padding='same')(d4)
    d3 = conv_block(layers.Concatenate()([u3, c3]), 256)
    u2 = layers.Conv2DTranspose(128, 2, strides=2, padding='same')(d3)
    d2 = conv_block(layers.Concatenate()([u2, c2]), 128)
    u1 = layers.Conv2DTranspose(64, 2, strides=2, padding='same')(d2)
    d1 = conv_block(layers.Concatenate()([u1, c1]), 64)
    out = layers.Conv2D(1, 1, activation='sigmoid')(d1)
    return Model(inputs=inp, outputs=out, name='UNet')

unet = build_unet()
unet.compile(optimizer=keras.optimizers.Adam(LR), loss=combined_loss, metrics=[dice_coef, iou_metric])
print(f'U-Net built with {unet.count_params():,} parameters.')


In [ ]:
# Paired LGG segmentation slices with streaming tf.data (zero memory crash)
all_tifs = glob(os.path.join(DATA_ROOT, '**', '*.tif'), recursive=True)
img_tifs = [p for p in all_tifs if '_mask' not in p]
paired_imgs, paired_masks = [], []
for p in img_tifs:
    mask_p = p.replace('.tif', '_mask.tif')
    if os.path.exists(mask_p):
        paired_imgs.append(p)
        paired_masks.append(mask_p)

paired_imgs = np.array(paired_imgs)
paired_masks = np.array(paired_masks)
print(f'Total paired slices found: {len(paired_imgs)}')

p_img_tr, p_img_val, p_mk_tr, p_mk_val = train_test_split(
    paired_imgs, paired_masks, test_size=0.15, random_state=SEED
)
print(f'Train slices: {len(p_img_tr)} | Val slices: {len(p_img_val)}')

def load_seg_pair(ip_t, mp_t):
    def _load(ip, mp):
        im = cv2.imread(ip.numpy().decode('utf-8'))
        mk = cv2.imread(mp.numpy().decode('utf-8'), cv2.IMREAD_GRAYSCALE)
        if im is None: im = np.zeros((*SEG_SIZE, 3), dtype=np.float32)
        else: im = cv2.resize(cv2.cvtColor(im, cv2.COLOR_BGR2RGB), SEG_SIZE).astype(np.float32) / 255.0
        if mk is None: mk = np.zeros((*SEG_SIZE, 1), dtype=np.float32)
        else: mk = (cv2.resize(mk, SEG_SIZE) > 127).astype(np.float32)[..., np.newaxis]
        return im, mk
    im, mk = tf.py_function(_load, [ip_t, mp_t], [tf.float32, tf.float32])
    im.set_shape((*SEG_SIZE, 3))
    mk.set_shape((*SEG_SIZE, 1))
    return im, mk

AUTOTUNE = tf.data.AUTOTUNE
seg_train_ds = (tf.data.Dataset.from_tensor_slices((p_img_tr, p_mk_tr))
                .shuffle(1000, seed=SEED)
                .map(load_seg_pair, num_parallel_calls=AUTOTUNE)
                .batch(SEG_BATCH).prefetch(AUTOTUNE))

seg_val_ds = (tf.data.Dataset.from_tensor_slices((p_img_val, p_mk_val))
              .map(load_seg_pair, num_parallel_calls=AUTOTUNE)
              .batch(SEG_BATCH).prefetch(AUTOTUNE))

cbs_unet = [
    keras.callbacks.ModelCheckpoint(UNET_MODEL_PATH, monitor='val_dice_coef', mode='max', save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor='val_dice_coef', mode='max', patience=8, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1)
]

print('Training U-Net model...')
h_seg = unet.fit(seg_train_ds, validation_data=seg_val_ds, epochs=SEG_EPOCHS, callbacks=cbs_unet)
print('U-Net training complete. Model saved to Drive.')


## 8. Segmentation Visualizations


In [ ]:
best_unet = tf.keras.models.load_model(UNET_MODEL_PATH, custom_objects={'combined_loss': combined_loss, 'dice_coef': dice_coef, 'iou_metric': iou_metric})

    # Select 5 validation samples that contain tumors for overlay display
    sample_imgs, sample_gts = [], []
    for ip, mp in zip(p_img_val, p_mk_val):
        mk = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
        if mk is not None and (mk > 127).sum() > 200:
            im = cv2.resize(cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB), SEG_SIZE)
            sample_imgs.append(im.astype(np.float32) / 255.0)
            sample_gts.append((cv2.resize(mk, SEG_SIZE) > 127).astype(np.uint8))
            if len(sample_imgs) == 5:
                break

    sample_imgs = np.array(sample_imgs, dtype=np.float32)
    preds = best_unet.predict(sample_imgs, batch_size=5)

    fig, axes = plt.subplots(len(sample_imgs), 3, figsize=(12, 4 * len(sample_imgs)))
    titles = ['Original MRI', 'Ground Truth Mask (Green)', 'U-Net Prediction Overlay (Red)']
    for ax, t in zip(axes[0], titles):
        ax.set_title(t, fontweight='bold', fontsize=12)

    for i in range(len(sample_imgs)):
        im = (sample_imgs[i] * 255).astype(np.uint8)
        gt = sample_gts[i]
        pr = (preds[i, ..., 0] > 0.5).astype(np.uint8)

        gt_ov = im.copy()
        gt_ov[gt == 1] = [50, 220, 50]
        gt_blend = cv2.addWeighted(im, 0.6, gt_ov, 0.4, 0)

        pr_ov = im.copy()
        pr_ov[pr == 1] = [220, 50, 50]
        pr_blend = cv2.addWeighted(im, 0.6, pr_ov, 0.4, 0)

        axes[i, 0].imshow(im); axes[i, 0].axis('off')
        axes[i, 1].imshow(gt_blend); axes[i, 1].axis('off')
        axes[i, 2].imshow(pr_blend); axes[i, 2].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(PREDS_DIR, 'segmentation_overlays.png'), dpi=150)
plt.show()
print('Segmentation overlays saved to Drive.')


## 9. Summary and Export

The trained models are saved in your Google Drive under:
`MyDrive/brain_tumor_project/models/`

Files generated:
1. `classifier_efficientnet.keras`
2. `unet_segmentation.keras`

Copy these files to your local `models/saved/` directory to run the local application:
```bash
streamlit run app/app.py
```
